## Final Integrated Stock Analysis App
This app combines data retrieval, technical plotting, ratio analysis, and a Net-Debt adjusted DCF valuation.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- UI Setup ---
ticker_input = widgets.Text(value='AAPL', description='Ticker:')
flow_type = widgets.Dropdown(options=[('Free Cash Flow', 'Free Cash Flow'), ('Operating Cash Flow', 'Operating Cash Flow')], value='Free Cash Flow', description='Cash Flow:')
growth_input = widgets.FloatSlider(value=0.08, min=0, max=0.5, step=0.01, description='Growth Y1-5:')
discount_input = widgets.FloatSlider(value=0.10, min=0.05, max=0.20, step=0.005, description='WACC:')
terminal_input = widgets.FloatSlider(value=0.02, min=0, max=0.05, step=0.005, description='Terminal G:')
btn = widgets.Button(description='Analyze Stock', button_style='primary', layout={'width': '300px'})
out = widgets.Output()

def get_val(df, keys, default=0):
    if df is None or df.empty: return default
    available = [str(k).strip() for k in df.index]
    for k in keys:
        if k in available:
            val = df.loc[k]
            if isinstance(val, pd.Series):
                valid_vals = val.dropna()
                return valid_vals if not valid_vals.empty else default
            return val
    return default

def run_app(b):
    with out:
        clear_output()
        ticker_symbol = ticker_input.value.upper()
        try:
            stock = yf.Ticker(ticker_symbol)
            hist_max = stock.history(period='max')
            hist_1y = stock.history(period='1y')
            if hist_1y.empty or hist_max.empty: raise ValueError('No data found for this ticker.')

            bs, cf, inc, info = stock.balance_sheet, stock.cashflow, stock.financials, stock.info

            # Determine currency for conversion
            stock_currency = info.get('currency', 'USD') # Default to USD if not found
            is_pence_currency = (stock_currency == 'GBp') # Check if currency is in pence

            # Define currency conversion factor
            currency_conversion_factor = 0.01 if is_pence_currency else 1.0

            # Determine currency symbol for display
            if stock_currency == 'USD':
                currency_symbol = '$'
            elif stock_currency == 'GBP' or stock_currency == 'GBp':
                currency_symbol = '£'
            else:
                currency_symbol = stock_currency + ' '

            # Apply currency conversion for current price
            curr_price_series = hist_1y['Close'].dropna()
            if not curr_price_series.empty:
                curr_price_raw = curr_price_series.iloc[-1]
                curr_price = curr_price_raw * currency_conversion_factor
            else:
                curr_price = np.nan

            # --- DCF Valuation Logic ---
            base_flow_raw = get_val(cf, [flow_type.value, 'Free Cash Flow', 'Operating Cash Flow', 'Net Income From Continuing Operations'])
            base_flow = (base_flow_raw.iloc[0] if isinstance(base_flow_raw, pd.Series) else base_flow_raw) * currency_conversion_factor
            shares_raw = get_val(bs, ['Ordinary Shares Number', 'Share Issued'])
            shares = shares_raw.iloc[0] if isinstance(shares_raw, pd.Series) else shares_raw
            if shares == 0 or pd.isna(shares): shares = info.get('sharesOutstanding', 1)
            cash_raw = get_val(bs, ['Cash And Cash Equivalents', 'Cash Cash Equivalents And Short Term Investments', 'Cash Financial'])
            cash = (cash_raw.iloc[0] if isinstance(cash_raw, pd.Series) else cash_raw) * currency_conversion_factor
            debt_raw = get_val(bs, ['Total Debt', 'Net Debt', 'Long Term Debt'])
            debt = (debt_raw.iloc[0] if isinstance(debt_raw, pd.Series) else debt_raw) * currency_conversion_factor

            dcf_steps = []
            temp_flow = base_flow
            for i in range(1, 6):
                temp_flow *= (1 + growth_input.value)
                pv = temp_flow / ((1 + discount_input.value)**i)
                dcf_steps.append({'Year': f'Year {i}', 'Projected Flow': temp_flow, 'Present Value': pv})

            df_steps = pd.DataFrame(dcf_steps)
            terminal_val = (temp_flow * (1 + terminal_input.value) / (discount_input.value - terminal_input.value))
            pv_tv = terminal_val / ((1 + discount_input.value)**5)
            enterprise_value = df_steps['Present Value'].sum() + pv_tv
            equity_value = enterprise_value + cash - debt
            fair_val = equity_value / shares

            print(f'=== COMPREHENSIVE ANALYSIS: {ticker_symbol} ===')

            # Handle division by zero or NaN for diff_pct if curr_price is not available or 0
            if pd.isna(curr_price) or curr_price == 0:
                diff_pct = np.nan
                rec = "N/A"
            else:
                diff_pct = (fair_val - curr_price) / curr_price * 100
                rec = "BUY" if diff_pct > 10 else "SELL" if diff_pct < -10 else "HOLD"

            val_params = pd.DataFrame({
                'Metric': ['Current Price', 'Fair Value (DCF)', 'Upside/Downside', 'Recommendation', 'Enterprise Value', 'Cash & Equivalents', 'Total Debt', 'Equity Value', 'Shares Outstanding', 'Terminal Value'],
                'Value': [
                    f'{currency_symbol}{curr_price:,.2f}' if pd.notna(curr_price) else 'N/A',
                    f'{currency_symbol}{fair_val:,.2f}',
                    f'{diff_pct:+.2f}%' if pd.notna(diff_pct) else 'N/A',
                    rec,
                    f'{currency_symbol}{enterprise_value:,.0f}',
                    f'{currency_symbol}{cash:,.0f}',
                    f'{currency_symbol}{debt:,.0f}',
                    f'{currency_symbol}{equity_value:,.0f}',
                    f'{shares:,.0f}', # Shares should not have a currency symbol
                    f'{currency_symbol}{terminal_val:,.0f}'
                ]
            })
            display(val_params.style.hide(axis='index'))

            # Displaying last 5 years of Operating Cash Flow
            print("\n--- Last 5 Years of Operating Cash Flow ---")
            # Assuming 'Operating Cash Flow' is a direct key in cf or can be derived
            # Taking the last 5 available periods for Operating Cash Flow
            op_cash_flow = get_val(cf, ['Operating Cash Flow']) * currency_conversion_factor
            if isinstance(op_cash_flow, pd.Series) and not op_cash_flow.empty:
                display(op_cash_flow.head(5).to_frame(name='Operating Cash Flow').style.format(f'{currency_symbol}{{:,.0f}}'))
            else:
                print("Operating Cash Flow data not available.")

            # Displaying DCF Steps (Projected Flows)
            print("\n--- Projected Cash Flows (Years 1-5) and Present Values ---")
            display(df_steps.style.format({'Projected Flow': f'{currency_symbol}{{:,.0f}}', 'Present Value': f'{currency_symbol}{{:,.0f}}'}))

            # Displaying Terminal Value Calculation Breakdown
            print("\n--- Terminal Value Calculation Breakdown ---")
            print(f"Last Projected Cash Flow (Year 5): {currency_symbol}{temp_flow:,.0f}") # Changed currency_flow to currency_symbol
            print(f"Terminal Growth Rate: {terminal_input.value:.2%}")
            print(f"Discount Rate (WACC): {discount_input.value:.2%}")
            print(f"Terminal Value (TV = Y5 Flow * (1 + Terminal G) / (WACC - Terminal G)): {currency_symbol}{terminal_val:,.0f}")
            print(f"Present Value of Terminal Value (PV_TV = TV / (1 + WACC)^5): {currency_symbol}{pv_tv:,.0f}")

            common_y = inc.columns.intersection(bs.columns)
            eff_df = pd.DataFrame() # Initialize eff_df
            pe_df = pd.DataFrame() # Initialize pe_df

            if not common_y.empty:
                print("\n--- Historical Efficiency & Leverage Summary ---")
                metrics_df = pd.DataFrame(index=common_y)
                metrics_df['Net Income'] = (inc.loc['Net Income'] if 'Net Income' in inc.index else 0) * currency_conversion_factor
                metrics_df['Revenue'] = (inc.loc['Total Revenue'] if 'Total Revenue' in inc.index else 1) * currency_conversion_factor
                metrics_df['Equity'] = (bs.loc['Stockholders Equity'] if 'Stockholders Equity' in bs.index else 1) * currency_conversion_factor
                metrics_df['EBIT'] = (inc.loc['EBIT'] if 'EBIT' in inc.index else 0) * currency_conversion_factor
                metrics_df['Pretax'] = (inc.loc['Pretax Income'] if 'Pretax Income' in inc.index else 1) * currency_conversion_factor
                metrics_df['TaxProv'] = (inc.loc['Tax Provision'] if 'Tax Provision' in inc.index else 0) * currency_conversion_factor
                metrics_df['Debt'] = (bs.loc['Total Debt'] if 'Total Debt' in bs.index else 0) * currency_conversion_factor
                metrics_df['Cash'] = (bs.loc['Cash And Cash Equivalents'] if 'Cash And Cash Equivalents' in bs.index else 0) * currency_conversion_factor

                h_roe = (metrics_df['Net Income'] / metrics_df['Equity']) * 100
                h_margin = (metrics_df['Net Income'] / metrics_df['Revenue'].replace(0, 1)) * 100
                h_de = metrics_df['Debt'] / metrics_df['Equity'].replace(0, 1)
                h_tax_rate = (metrics_df['TaxProv'] / metrics_df['Pretax']).fillna(0.21)
                h_invested = metrics_df['Equity'] + metrics_df['Debt'] - metrics_df['Cash']
                h_roic = (metrics_df['EBIT'] * (1 - h_tax_rate)) / h_invested.replace(0, 1) * 100

                eff_df = pd.DataFrame({'ROE (%)': h_roe, 'ROIC (%)': h_roic, 'Net Margin (%)': h_margin, 'Debt/Equity': h_de}).dropna()
                if not eff_df.empty:
                    display(eff_df.style.format({'ROE (%)': '{:.2f}%', 'ROIC (%)': '{:.2f}%', 'Net Margin (%)': '{:.2f}%', 'Debt/Equity': '{:.2f}'}))

                print("\n--- Historical P/E Ratio (Last 10 Years) ---")
                pe_data = []
                hist_px = hist_max['Close'].copy()
                if hist_px.index.tz is None: hist_px.index = hist_px.index.tz_localize('UTC')

                for d in sorted(common_y, reverse=True)[:10]:
                    try:
                        dt_lookup = d.tz_localize('UTC') if d.tzinfo is None else d.tz_convert('UTC')
                        ni = inc.loc['Net Income', d] * currency_conversion_factor
                        sh = shares_raw.loc[d] if isinstance(shares_raw, pd.Series) and d in shares_raw.index else shares
                        idx_loc = hist_px.index.get_indexer([dt_lookup], method='pad')[0]
                        px = hist_px.iloc[idx_loc] * currency_conversion_factor if idx_loc != -1 else np.nan

                        if not pd.isna(px) and sh > 0 and ni != 0:
                            eps = ni / sh
                            pe_data.append({'Year': d.year, 'EPS': eps, 'Price': px, 'PE': px / eps})
                    except: continue

                pe_df = pd.DataFrame(pe_data)
                if not pe_df.empty:
                    pe_df = pe_df.sort_values('Year')
                    display(pe_df.set_index('Year').style.format({'EPS': f'{currency_symbol}{{:,.2f}}', 'Price': f'{currency_symbol}{{:,.2f}}', 'PE': '{:.2f}'}))

            # --- Visual Suite (Now 5 Panels) ---
            fig, axes = plt.subplots(5, 1, figsize=(12, 35))
            axes[0].plot(hist_1y.index, hist_1y['Close'] * currency_conversion_factor)
            axes[0].set_title('Price Trend (1Y)')

            if not pe_df.empty:
                axes[1].plot(pe_df['Year'].astype(str), pe_df['PE'], marker='o', color='purple')
                axes[1].set_title('Historical P/E Ratio')
            else:
                axes[1].text(0.5, 0.5, 'No P/E data', ha='center')

            if not eff_df.empty:
                axes[2].plot(eff_df.index, eff_df['ROE (%)'], label='ROE', marker='o')
                axes[2].plot(eff_df.index, eff_df['ROIC (%)'], label='ROIC', marker='s')
                axes[2].plot(eff_df.index, eff_df['Net Margin (%)'], label='Net Margin', marker='x', linestyle='--')
                axes[2].legend(); axes[2].set_title('Efficiency & Profitability Trends')

                axes[3].bar(eff_df.index.strftime('%Y'), eff_df['Debt/Equity'], color='brown', alpha=0.7)
                axes[3].set_title('Debt-to-Equity Leverage Trend')
                axes[3].set_ylabel('Ratio')
            else:
                axes[2].text(0.5, 0.5, 'No efficiency data', ha='center')
                axes[3].text(0.5, 0.5, 'No leverage data', ha='center')

            hmp = hist_max.reset_index().dropna(subset=['Close'])
            hmp['Date_Ord'] = hmp['Date'].map(pd.Timestamp.toordinal)
            sns.regplot(x='Date_Ord', y='Close', data=hmp, scatter_kws={'alpha':0.1}, ax=axes[4])
            axes[4].set_title('Long Term Trend')
            plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'Error: {e}')

btn.on_click(run_app)
display(widgets.VBox([widgets.HBox([ticker_input, flow_type]), widgets.HBox([growth_input, discount_input, terminal_input]), btn, out]))

In [ ]:
# Set the ticker_input value to 'AMZN' and run the analysis to test the current price display
ticker_input.value = 'AMZN'
run_app(None)